In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas", "numpy", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy().reset_index(drop=True)

print({
    "num_examples": len(df),
    "columns": df.columns.tolist(),
    "label_min": float(df["label"].min()),
    "label_max": float(df["label"].max()),
    "label_mean": float(df["label"].mean()),
})
print(df.head(10))


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

print({
    "embedding_shape_sentence1": tuple(emb1.shape),
    "embedding_shape_sentence2": tuple(emb2.shape),
    "cosine_min": float(cosine_similarity.min()),
    "cosine_max": float(cosine_similarity.max()),
    "predicted_min": float(predicted_score_0_5.min()),
    "predicted_max": float(predicted_score_0_5.max()),
})


In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
absolute_error = np.abs(predicted_score_0_5 - labels)
squared_error = (predicted_score_0_5 - labels) ** 2
mae = float(np.mean(absolute_error))
rmse = float(np.sqrt(np.mean(squared_error)))

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error
results_df["squared_error"] = squared_error

largest_errors_df = results_df.sort_values(
    by=["absolute_error", "squared_error"],
    ascending=[False, False]
).reset_index(drop=True)

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "absolute_error"]].head(10))
print(largest_errors_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error"]].head(10))


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"mean_absolute_error: {mae:.6f}")
print(f"root_mean_squared_error: {rmse:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")

top_error_examples = largest_errors_df[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "absolute_error"
]].head(10)
print(top_error_examples.to_dict(orient="records"))
